# Модуль 06 · Урок 4. Parquet, Arrow и выбор движка обработки

**Как проходить:** изучите все разделы теории последовательно. Запустите демонстрационные ячейки, осознайте входы и выходы методов. Только после этого решайте задания. В условиях используются другие предметные области и комбинации действий — готовых решений в теории нет.

**Обещание урока:** все необходимые операции объяснены ниже; внешний поиск не нужен.

**Стандарт проверки:** правильные значения, форма, порядок и неизменность входа; минимум два набора данных на задачу.

In [ ]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
import numpy as np
from tests.notebook_checks import expected_for, check_result, show_progress, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


## 1. Текстовый CSV и колоночный Parquet
CSV хранит строки текста, не включает полную информацию о типах, а некоторые значения требуют экранирования. Parquet — бинарный **колоночный** формат: значения одного столбца хранятся вместе, обычно лучше сжимаются и позволяют считывать только нужные колонки. Он полезен для аналитических хранилищ и многократного чтения данных. Для `to_parquet`/`read_parquet` нужен установленный движок, обычно `pyarrow`.

In [ ]:
import pandas as pd
frame = pd.DataFrame({'year':[2024,2025],'country':['RU','KZ'],'revenue':[10.0,15.0]})
# Для выполнения этой ячейки должен быть установлен pyarrow.
# frame.to_parquet('example.parquet',index=False)
# restored = pd.read_parquet('example.parquet',columns=['country','revenue'])

## 2. Arrow и типы данных
Apache Arrow задаёт формат колонок в памяти и помогает обмениваться данными между системами. `dtype_backend='pyarrow'` в некоторых функциях чтения pandas возвращает Arrow-backed типы. Это может уменьшать память и улучшать совместимость, но не означает автоматического ускорения **всех** операций. Сравнивайте результаты и производительность на своих задачах. В pandas 3.x строковый dtype стал отдельным типом и может использовать Arrow.

## 3. Схема данных и партиционирование
Parquet-файлы часто разделяют по дате/стране/другому фильтрующему признаку. Такой набор файлов называется partitioned dataset. Выбор колонок и фильтров при чтении снижает объём ввода-вывода. Однако создание миллионов крошечных файлов может ухудшить работу; размер партиций также нужно проектировать.

## 4. Выбор между pandas, SQL, DuckDB, Polars, Dask
pandas удобен для аналитики в памяти, сложных исследований и интеграции с Python. SQL-движки хорошо выполняют фильтрации и агрегации рядом с данными; DuckDB работает эффективно с локальными Parquet и CSV; Polars ориентирован на колоночную обработку и lazy execution; Dask распределяет части задач на несколько процессов или машин. Не переносите задачу на новый стек только из-за одного большого файла: сначала измерьте узкое место.

**Проверьте себя:** какой формат лучше сохраняет типы? Почему только нужные колонки ускоряют чтение? В каком случае агрегирование лучше выполнить в SQL до загрузки в pandas?

---
## Самостоятельная практика

**Инструкция:** выполните ячейку исходных данных, напишите обычный код в следующей ячейке и сохраните итог в `result`. Под решением показан ожидаемый вывод. Затем запустите проверку. **Не нужно писать функции.**

### m6_16 · schema report

Получите dict, где ключ — название столбца, значение — строковое представление dtype (`str(dtype)`).

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m6_16: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m6_16 = expected_for("m6_16", df=df)


In [ ]:
# m6_16: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

Словарь (`dict`) с указанными ключами и значениями.

```python
{'order_id': 'object',
 'client_id': 'int64',
 'city': 'object',
 'category': 'object',
 'price': 'float64',
 'qty': 'int64',
 'discount': 'int64',
 'status': 'object',
 'date': 'object',
 'comment': 'object'}
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m6_16", result, _expected_m6_16)

### m6_17 · size kib

Получите оценку общего объёма DataFrame (включая индекс, deep=True) в КиБ, разделив байты на 1024.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m6_17: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m6_17 = expected_for("m6_17", df=df)


In [ ]:
# m6_17: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

Результат — числовая оценка размера памяти, зависящая от версии pandas и представления строк.

`float`. Результат для показанного примера: `2.763671875`.

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m6_17", result, _expected_m6_17)

### m6_18 · project columns

Получите независимый DataFrame только с selected в указанном порядке, как если бы выбирали колонки из колоночного файла.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.
- `selected` — список названий столбцов; тип: `list`.


**Выполнение:** используйте готовые переменные `df`, `selected`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m6_18: используются переменные df, selected

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

selected = ['city', 'price', 'qty']

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m6_18 = expected_for("m6_18", df=df, selected=selected)


In [ ]:
# m6_18: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df, selected
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 3 столбца; колонки: `city`, `price`, `qty`.

```text
     city  price  qty
z  Москва  127.0    2
b   Тверь  247.0    1
a  Москва   97.0    4
d  Казань  317.0    1
x   Тверь   57.0    3
q  Казань  147.0    0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m6_18", result, _expected_m6_18)

### m6_19 · partition by month

Получите dict, где ключи — строки вида `YYYY-MM`, значения — DataFrame соответствующих заказов в исходном порядке с исходными индексами.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m6_19: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m6_19 = expected_for("m6_19", df=df)


In [ ]:
# m6_19: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

Словарь (`dict`) с указанными ключами и значениями.

Ключи: `2026-01`, `2026-02`. Каждый элемент словаря — отдельная таблица/серия.

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m6_19", result, _expected_m6_19)

### m6_20 · parquet ready

Подготовьте данные к колоночному хранению: получите DataFrame из колонок `order_id`,`client_id`,`city`,`revenue`; revenue = price * qty * (1-discount/100), индексы сбросьте.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m6_20: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m6_20 = expected_for("m6_20", df=df)


In [ ]:
# m6_20: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 4 столбца; колонки: `order_id`, `client_id`, `city`, `revenue`.

```text
  order_id  client_id    city  revenue
0  demo_o1          1  Москва    254.0
1  demo_o2          2   Тверь    222.3
2  demo_o3          1  Москва    368.6
3  demo_o4          3  Казань    317.0
4  demo_o5          2   Тверь    136.8
5  demo_o6          4  Казань      0.0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m6_20", result, _expected_m6_20)

---
## Итоги урока

Если все проверки пройдены, повторите своими словами: **что делает каждый метод, какие параметры важны и какая ошибка наиболее вероятна**. Перед переходом к следующему уроку попробуйте решить одно задание заново с чистой ячейки. Автотест подтверждает поведение на известных наборах, но не заменяет код-ревью.

In [ ]:
show_progress()